# Data Collection & Preparation

![image-3.png](attachment:image-3.png)

## 1.Import python packages

In [0]:
import pandas as pd
import numpy as np

import os

# path=os.environ['USERPROFILE']+r'/OneDrive/BDA2/'
path=os.environ['USERPROFILE']+r'/Documents/BDA2/'

file_path=path
# file_path=os.environ['USERPROFILE']+'/Business_Data_Analysis/'

## 2.Establish connection to database

![image-2.png](attachment:image-2.png)

In [0]:
import pyodbc
import urllib
import sqlalchemy

'''sources databases'''
params_p = urllib.parse.quote_plus("DRIVER={SQL Server Native Client 11.0};"
                                 "SERVER=localhost\SQLEXPRESS;"
                                 "DATABASE=DataWarehouse1;"
                                 "UID=sa;"
                                 "PWD=user1")

engine_p = sqlalchemy.create_engine("mssql+pyodbc:///?odbc_connect={}".format(params_p))


params_s = urllib.parse.quote_plus("DRIVER={SQL Server Native Client 11.0};"
                                 "SERVER=localhost\SQLEXPRESS;"
                                 "DATABASE=DataWarehouse2;"
                                 "UID=sa;"
                                 "PWD=user1")

engine_s = sqlalchemy.create_engine("mssql+pyodbc:///?odbc_connect={}".format(params_s))



'''destination databases'''

params_datahub = urllib.parse.quote_plus("DRIVER={SQL Server Native Client 11.0};"
                                 "SERVER=localhost\SQLEXPRESS;"
                                 "DATABASE=datahub;"
                                 "UID=sa;"
                                 "PWD=user1")

engine_datahub = sqlalchemy.create_engine("mssql+pyodbc:///?odbc_connect={}".format(params_datahub))

## 3.back up your data

![image.png](attachment:image.png)

In [0]:
'''get historical data from source database'''
df_sales_p=pd.read_sql_table('user_table',engine_p)

In [0]:
from datetime import datetime
dateTimeObj = datetime.now()
timestampStr = dateTimeObj.strftime("%d-%m-%Y-%H-%M-%S")
print('Current Timestamp : ', timestampStr)

df_sales_p.to_csv(file_path+'data/user_table_backup_'+timestampStr+'.csv',index=False)

In [0]:
from datetime import datetime
dateTimeObj = datetime.now()
timestampStr = dateTimeObj.strftime("%d-%m-%Y-%H-%M-%S")
print('Current Timestamp : ', timestampStr)

## 4.ETL data to datahub

![image.png](attachment:image.png)

In [0]:
df_sales_s=pd.read_sql('home_page_table',engine_s)

df_sales_s.to_sql("home_page_table",engine_datahub,if_exists='replace',index=False)

## a. Append new data to historical data

### i. Option 1: using python

![image.png](attachment:image.png)

In [0]:
from datetime import datetime


#get new data

df_sales_p_new=pd.read_csv(file_path+'data/user_table_new.csv',index_col=False)


In [0]:
df_sales_p_new

In [0]:
'''get historical data from source database'''
df_sales_p=pd.read_sql_table('user_table',engine_p)
df_sales_p.shape

In [0]:
'''append new data to historical data'''
df_sales_p=pd.concat([df_sales_p,df_sales_p_new])
df_sales_p.shape

In [0]:
help(pd.concat)

In [0]:
'''drop duplicate'''
df_sales_p.drop_duplicates(keep='first', inplace=True)

df_sales_p

### ii. Check data

In [0]:
df_sales_p_new.shape,df_sales_p.shape

In [0]:
df_sales_p.tail()

### iii. Export updated historical data to database

In [0]:
df_sales_p.to_sql("user_table", engine_datahub,if_exists='replace',index=False)

### i. Option 2: using SQL: passthrough

![image-2.png](attachment:image-2.png)

In [0]:
# from datetime import datetime
# dateparse = lambda x: datetime.strptime(x, '%m/%d/%Y')


'''get new data'''
df_sales_p_new=pd.read_csv(file_path+'data/user_table_new.csv', index_col=False)


'''export new data into datahub'''
df_sales_p_new.to_sql("user_table_new", engine_datahub,if_exists='replace',index=False) 

In [0]:
#append new data to historical data

server = 'localhost\SQLEXPRESS' 
database = 'datahub' 
username = 'sa' 
password = 'user1' 
cnxn = pyodbc.connect('DRIVER={SQL Server};SERVER='+server+';DATABASE='+database+';UID='+username+';PWD='+ password)
cursor = cnxn.cursor()

sql='''

DROP TABLE IF EXISTS #temp1

select * into #temp1
from (
select * from [dbo].[sql_user_table]
union 
select * from [dbo].[sql_user_table]) a

drop table if EXISTS [dbo].[sql_user_table]

select * into [dbo].[sql_user_table]
from #temp1

'''

cursor.execute(sql)
cnxn.commit()
cursor.close()